In [ ]:
# ═════════════════════════════════════════════════════════════
#  MOE STICKER BOT — ALL-IN-ONE CELL (No UI boxes, full ANSI)
# ═════════════════════════════════════════════════════════════

# ==================== CONFIGURATION (EDIT HERE) ====================
BOT_TOKEN = "8760934506:AAF2U0ZInHED8r7V_Exz-z8SeIqxLszCKMU"                     # <-- YOUR TELEGRAM BOT TOKEN
ENABLE_DB = False                  # Set True if you have a MySQL DB
DB_ADDR = "localhost:3306"
DB_USER = "moe_bot"
DB_PASS = ""
DB_NAME = "moe_sticker_bot"
ENABLE_WEBAPP = False              # Set True to expose WebApp via ngrok
WEBAPP_PORT = 8080
NGROK_AUTHTOKEN = ""               # Required if ENABLE_WEBAPP=True
DATA_DIR = "moe_sticker_bot_data"
LOG_LEVEL = "info"                 # debug, info, warn, error
HTTP_PROXY = ""                    # Optional http://proxy:port
# ===================================================================

import sys, time, subprocess, os, urllib.request, json, requests, threading, signal
from itertools import cycle
from tqdm.notebook import tqdm

# ---------- ANSI Color Class ----------
class C:
    R = '\033[0m'; B = '\033[1m'; D = '\033[2m'
    BLK = '\033[30m'; RD = '\033[31m'; GN = '\033[32m'; YL = '\033[33m'
    BL = '\033[34m'; MG = '\033[35m'; CY = '\033[36m'; WH = '\033[37m'
    BRD = '\033[91m'; BGN = '\033[92m'; BYL = '\033[93m'; BBL = '\033[94m'
    BMG = '\033[95m'; BCY = '\033[96m'; BWH = '\033[97m'
    BGRD = '\033[41m'; BGGN = '\033[42m'; BGYL = '\033[43m'; BGBL = '\033[44m'

def success(m): print(f"{C.B}{C.BGGN}{C.BLK} ✓ {m} {C.R}")
def error(m):   print(f"{C.B}{C.BGRD}{C.WH} ✗ {m} {C.R}")
def info(m):    print(f"{C.B}{C.BGBL}{C.WH} ℹ {m} {C.R}")
def warn(m):    print(f"{C.B}{C.BGYL}{C.BLK} ⚠ {m} {C.R}")
def header(t):  print(f"\n{C.B}{C.BCY}{'═'*50}\n  {t}\n{'═'*50}{C.R}\n")

def spinner(msg, dur=2):
    frames = cycle(['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'])
    end = time.time() + dur
    while time.time() < end:
        sys.stdout.write(f'\r{C.BCY}{next(frames)} {msg}{C.R}  ')
        sys.stdout.flush()
        time.sleep(0.1)
    sys.stdout.write(f'\r{C.BGN}✔{C.R} {msg}   \n')

# ---------- 1. Setup & Dependencies ----------
header("System Dependencies")
spinner("Updating packages", 1)
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
success("Core packages installed")

# Go language
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
print(f"{C.CY}⬇ Downloading Go...{C.R}")
with tqdm(unit='B', unit_scale=True, desc=f"{C.BCY}Go{C.R}") as t:
    urllib.request.urlretrieve(url, "go.tar.gz", reporthook=lambda b,bs,total: t.update(b*bs-t.n))
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH
success(f"Go {subprocess.getoutput('go version').split()[2]} installed")

# Python helpers
header("Installing Python Helpers")
helpers = [("msb_emoji.py","Emoji"), ("msb_kakao_decrypt.py","Kakao"), ("msb_rlottie.py","Lottie")]
for f,desc in helpers:
    !wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/{f} -O /usr/local/bin/{f}
    !chmod +x /usr/local/bin/{f}
    print(f"  {C.GN}✓{C.R} {desc}")
success("Helpers installed")

# Build bot from source
header("Building MoeStickersBot")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git 2>&1 | grep -v "Cloning"
%cd MoeStickersBot
spinner("Downloading Go modules", 2)
!go mod download
spinner("Compiling binary", 3)
!go build -o MoeStickersBot cmd/MoeStickersBot/main.go
if os.path.exists("MoeStickersBot"):
    sz = os.path.getsize("MoeStickersBot")/1024/1024
    success(f"Build complete — Binary: {sz:.1f} MB")
else:
    error("Build failed")
    sys.exit(1)

# ---------- 2. Launch Bot (with ngrok if enabled) ----------
if not BOT_TOKEN:
    error("BOT_TOKEN is not set! Edit the variable at the top of this cell and re-run.")
    sys.exit(1)

header("Launching Bot")

WEBAPP_URL = ""
ngrok_proc = None
if ENABLE_WEBAPP:
    if not NGROK_AUTHTOKEN:
        warn("WebApp enabled but no NGROK_AUTHTOKEN — disabling WebApp")
        ENABLE_WEBAPP = False
    else:
        if not os.path.exists("./ngrok"):
            !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
        !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
        !pkill -f ngrok || true
        ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        spinner("Starting ngrok", 3)
        for _ in range(10):
            try:
                r = requests.get("http://127.0.0.1:4040/api/tunnels")
                if r.status_code == 200:
                    tuns = r.json()['tunnels']
                    if tuns:
                        WEBAPP_URL = tuns[0]['public_url']
                        success(f"ngrok URL: {WEBAPP_URL}")
                        break
            except: pass
            time.sleep(1)
        else:
            error("Could not retrieve ngrok URL")
            ENABLE_WEBAPP = False

# Build command line
cmd_line = ["./MoeStickersBot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]
if ENABLE_DB and DB_ADDR:
    cmd_line.extend([f"--db_addr={DB_ADDR}", f"--db_user={DB_USER}", f"--db_pass={DB_PASS}", f"--db_name={DB_NAME}"])
if ENABLE_WEBAPP and WEBAPP_URL:
    cmd_line.append(f"--webapp_url={WEBAPP_URL}")
    cmd_line.append(f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}")
if HTTP_PROXY:
    cmd_line.append(f"--http_proxy={HTTP_PROXY}")

print(f"{C.D}Command: {' '.join(cmd_line).replace(BOT_TOKEN, '[REDACTED]')}{C.R}")
log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd_line, stdout=log_out, stderr=log_err)
spinner("Starting bot", 3)
time.sleep(2)

if process.poll() is None:
    success(f"Bot is RUNNING — PID {process.pid}")
    print(f"{C.B}{C.BGN}📱 Send /start to your bot on Telegram!{C.R}")
    if WEBAPP_URL:
        print(f"{C.B}{C.BCY}🌐 WebApp: {WEBAPP_URL}{C.R}")
else:
    error("Bot exited immediately. Check logs:")
    !cat bot_stderr.log
    sys.exit(1)

# ---------- 3. Interactive Monitor (no extra cells) ----------
def cleanup():
    print(f"\n{C.BGYL}Shutting down...{C.R}")
    if process.poll() is None:
        process.terminate()
        time.sleep(1)
        process.kill()
    if ngrok_proc and ngrok_proc.poll() is None:
        ngrok_proc.terminate()
    !pkill -f ngrok 2>/dev/null
    success("Cleanup done")

def show_logs(logtype, lines=30):
    fname = f"bot_{logtype}.log"
    if os.path.exists(fname):
        print(f"{C.BCY}📄 Last {lines} lines of {fname}:{C.R}\n")
        !tail -n {lines} {fname}
    else:
        warn(f"{fname} not found")

print(f"\n{C.B}{C.BCY}{'─'*50}")
print("  Interactive Control (type command)")
print("  logs   - show last 30 lines of stderr")
print("  logsout- show last 30 lines of stdout")
print("  stop   - terminate bot and ngrok")
print("  exit   - leave this menu (bot keeps running)")
print(f"{'─'*50}{C.R}\n")

while True:
    cmd = input(f"{C.B}{C.CY}❯ {C.R}").strip().lower()
    if cmd == "logs":
        show_logs("stderr")
    elif cmd == "logsout":
        show_logs("stdout")
    elif cmd == "stop":
        cleanup()
        break
    elif cmd == "exit":
        print(f"{C.D}Bot continues in background. Use 'pkill -f MoeStickersBot' to stop later.{C.R}")
        break
    else:
        print(f"{C.YL}Unknown command. Try: logs, logsout, stop, exit{C.R}")